# 04 — Operational summary

This notebook summarises the final process used for the deliverable.

For the full model-selection history — including P3–P14, MCMC, Empirical Bayes and the EB → parametric derivation of the operational priors — start with [00_model_selection_history.ipynb](00_model_selection_history.ipynb).

The final operational chain is:

1. Define a physical initial prior around the expected first-contact point with the water.
2. Execute a conservative first mission.
3. If there is no detection, update the posterior with the partial-failure likelihood.
4. Displace the search progressively along the physical drift direction.
5. Compare the three main witness hypotheses: centre, right and left.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings; warnings.filterwarnings('ignore')
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from update_version.simulator.grid import load_grid
from update_version.simulator.environment import SearchEnvironment
from update_version.simulator.true_detection import TrueDetector
from update_version.modeling.features import (
    ACCIDENT_POINT, V_PLANE, V_WIND, V_DRIFT, ALPHA_WIND,
    expected_landing, trajectory_axes, cell_distances, evaluate_pi,
)
from update_version.modeling.priors_logistic import PRIORS
from update_version.modeling.detection import DETECTORS
from update_version.strategies.next_mission import (
    hdr_mask, propose_via_hdr_topk, propose_drift_tail_rescue,
    progressive_drift_prior, progressive_scenario_prior,
    apply_drift_tail_escape, apply_progressive_scenario_escape,
    propose_progressive_scenario_search,
)

grid = load_grid('../data/grid_dataset.csv')
BUDGET_CAP = 530



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/josecalatayud/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/josecalatayud/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/josecalatayud/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 736, in start
    self.io_loop.star

AttributeError: _ARRAY_API not found

## The final model

The final proposal **does not force a specific search zone**. Instead it combines four physical hypotheses:

1. The aircraft follows the main trajectory.
2. The aircraft falls towards the right of the trajectory.
3. The aircraft falls towards the left of the trajectory.
4. The wind affects the fall more strongly than the canonical assumption.

Once the aircraft is in the water, if the early missions fail, the search advances along the physical drift direction.

The recommended strategy is `mixture`, because we do not know with certainty which witness best describes the real case.

In [ ]:
# Recommended operational configuration.
PRIOR_FINAL = 'P13_impact_robust_uniform10'
SEARCH_SCENARIO_FINAL = 'mixture'
DETECTOR_FINAL = 'D1_saturating_exponential'
UPDATE_TEMPERATURE = 1.5
print(PRIOR_FINAL, SEARCH_SCENARIO_FINAL, DETECTOR_FINAL, UPDATE_TEMPERATURE)

## Example run

We simulate a case in which the object may have fallen under any of the witness hypotheses.

The search strategy uses `mixture`, which is robust to four possibilities:

- centre,
- right,
- left,
- strong wind.

The aim is to see how the posterior and the search rectangles shift after each observation.

In [ ]:
# Reusable helpers for the final summary.
def direct_update(posterior, rec, detector, temperature=1.5):
    rho = np.asarray(detector.rho(grid.depth, grid.roughness, rec.effort))
    covered = grid.coverage_mask(rec.x_min, rec.x_max, rec.y_min, rec.y_max)
    q = np.where(covered, rho, 0.0)
    q = np.clip(q, 1e-12, 1 - 1e-12)
    L = (q if rec.s_t else 1 - q) ** temperature
    post = posterior * L
    return post / post.sum(), q, L, covered

def run_final_case(true_scenario='center', seed=1, max_missions=10):
    fake_history = [type('M', (), {'s_t': 0})() for _ in range(3)]
    pi_true, *_ = progressive_scenario_prior(grid, fake_history, scenario=true_scenario)
    rng = np.random.default_rng(seed)
    true_cell = int(rng.choice(grid.n_cells, p=pi_true))
    det = DETECTORS[DETECTOR_FINAL]
    env = SearchEnvironment(grid=grid, detector=TrueDetector(),
                             budget_total=BUDGET_CAP,
                             rng=np.random.default_rng(seed))
    env.plant_object(cell_id=true_cell)
    posterior = PRIORS[PRIOR_FINAL].prior_predictive_pi(grid.x, grid.y,
                                                          n_samples=900, seed=0)
    trace = [posterior.copy()]
    rows = []
    for _ in range(max_missions):
        if env.budget_remaining < 4:
            break
        prop = propose_progressive_scenario_search(
            posterior, grid, det, env.budget_remaining,
            history=env.history, scenario=SEARCH_SCENARIO_FINAL,
        )
        rec = env.run_mission(**prop.as_kwargs())
        post, q, L, covered = direct_update(posterior, rec, det, UPDATE_TEMPERATURE)
        if rec.s_t == 0:
            post, mix, center_long, center_xy, sc = apply_progressive_scenario_escape(
                post, grid, env.history, scenario=SEARCH_SCENARIO_FINAL,
            )
        else:
            mix = 0.0
            _, center_long, center_xy, _, sc = progressive_scenario_prior(
                grid, env.history, scenario=SEARCH_SCENARIO_FINAL,
            )
        rows.append({
            'mission': rec.mission_id,
            'x_min': rec.x_min, 'x_max': rec.x_max,
            'y_min': rec.y_min, 'y_max': rec.y_max,
            'effort': rec.effort, 'cost': rec.cost,
            's_t': rec.s_t, 'budget_remaining': rec.budget_remaining,
            'mix': mix, 'd_long_center': center_long,
            'center_x': center_xy[0], 'center_y': center_xy[1],
            'contains_true': bool(covered[true_cell]),
            'true_cell': true_cell,
            'true_x': grid.x[true_cell], 'true_y': grid.y[true_cell],
        })
        posterior = post
        trace.append(post.copy())
        if rec.s_t == 1:
            break
    return pd.DataFrame(rows), trace, env

final_df, final_trace, final_env = run_final_case(true_scenario='right', seed=3)
final_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, values, title in [
    (axes[0], final_trace[0],  'Initial prior'),
    (axes[1], final_trace[-1], 'Final posterior'),
    (axes[2], final_trace[-1], 'Missions and moving physical centres'),
]:
    ax.imshow(grid.reshape_2d(values), origin='lower',
               extent=[0, grid.Nx, 0, grid.Ny], cmap='magma')
    ax.set_title(title)
true_cell = int(final_df.true_cell.iloc[0])
for ax in axes:
    ax.scatter(grid.x[true_cell], grid.y[true_cell],
                color='lime', marker='X', s=90, edgecolor='black')
for _, r in final_df.iterrows():
    color = 'lime' if r.s_t == 1 else 'white'
    axes[2].add_patch(plt.Rectangle((r.x_min, r.y_min),
                                      r.x_max - r.x_min, r.y_max - r.y_min,
                                      edgecolor=color, facecolor='none', lw=1.5))
    axes[2].scatter(r.center_x, r.center_y, color='deepskyblue',
                      marker='P', s=70, edgecolor='black')
    axes[2].text(r.x_min, r.y_max + 0.2, str(int(r.mission)), color=color)
plt.tight_layout(); plt.show()

## Conclusions

The final strategy is consistent with the witness statements and with the physics of the problem.

We first search close to the expected impact point. If the mission fails, the posterior is updated using the partial-failure likelihood. Then the model displaces the search along physical hypotheses: centre, right, left, or strong wind.

The scenario mixture is the most robust option when we do not know which witness was the most accurate.

## Three-scenario decision summary

The mission can be interpreted as a comparison between three main hypotheses.

### If the bomb lay along the central drift

The search must advance along $d_{\mathrm{long}}$ while keeping the trajectory centred. This is the most conservative hypothesis: the aircraft kept its velocity and then drifted along the dominant direction in the water.

### If the bomb lay to the right of the drift

The search must advance along $d_{\mathrm{long}}$ but with the lateral centre shifted to the positive side of $d_{\mathrm{trans}}$. This represents the case in which the lateral witness observed a fall to the right of the trajectory.

### If the bomb lay to the left of the drift

The search must advance along $d_{\mathrm{long}}$ but with the lateral centre shifted to the negative side of $d_{\mathrm{trans}}$. This is the opposite lateral hypothesis.

The final `mixture` strategy combines the three lateral hypotheses and a small strong-wind component. That is why it is more robust when we do not know which witness was the most reliable.

In [5]:
# Compact comparison used in the final explanation.
scenarios_to_show = ['center', 'right', 'left']
rows = []
for scenario in scenarios_to_show:
    final_df, final_trace, final_env = run_final_case(true_scenario=scenario, seed=4)
    rows.append({
        'true_scenario': scenario,
        'detected': bool(len(final_df) and final_df.s_t.max() == 1),
        'n_missions': len(final_df),
        'cost_used': final_env.budget_used,
        'ever_covered_true': bool(len(final_df) and final_df.contains_true.any()),
        'true_cell': int(final_df.true_cell.iloc[0]) if len(final_df) else None,
        'true_x': float(final_df.true_x.iloc[0]) if len(final_df) else None,
        'true_y': float(final_df.true_y.iloc[0]) if len(final_df) else None,
    })
pd.DataFrame(rows)

,true_scenario,detected,n_missions,cost_used,ever_covered_true,true_cell,true_x,true_y
0,center,False,10,482.0,True,668,18.5,13.5
1,right,False,10,482.0,False,772,22.5,15.5
2,left,True,7,292.0,True,520,20.5,10.5
